# **Ejercicio 2: Entregable Final (videogames_reviews.csv)**

Vamos a trabajar con un dataframe que contiene información sobre reseñas de videojuegos. Cada fila del dataframe contiene la información de una reseña, y las columnas son las siguientes:
- Contenido: texto de la reseña
- Valoración: Recomendado o No recomendado
- Recomendado binario: 1 si la valoración es Recomendado, 0 si la valoración es No recomendado
  
El objetivo de este ejercicio es conseguir extraer información valiosa de las reseñas de un dataset más 'crudo' o que necesita un preprocesado mayor que el del ejercicio 1.

1. **Filtramos por las 100 reseñas con el contenido más largo**. Esto ya os lo doy hecho más abajo.
2. En el DataFrame resultante, habremos obtenido muchas filas en las que realmente el contenido de la reseña no es nada relevante. **Instruir vía Prompting a un LLM para que filtre las filas del DataFrame que de verdad aportan información de la que se pueden extraer insights relevantes**
3. **Paso final:** extraer información relevante de este último dataframe doblemente filtrado usando un segundo LLM. Por ejemplo, un json final como este:

    ```json
    {
    "Id": "{{id}}",
    "SentimientoGeneral": "Positivo|Negativo|Neutral",
    "AspectosPositivos": "[lista de aspectos positivos mencionados]",
    "AspectosNegativos": "[lista de aspectos negativos mencionados]",
    "Dificultad": "Demasiado Fácil|Fácil|Equilibrado|Difícil|Demasiado Difícil|No Mencionado",
    "Recomendado": "{{recomendado}}"
    }
    ```

**Consideraciones adicionales:**

- También tienes un csv de ejemplo de resultado final (analisis_videojuegos_resultados.csv) donde se han extraído otras entidades y se ha entregado otro formato. **Esto es a libre elección, pero un mínimo de 3 entidades extraídas** por el segundo LLM.
- Los modelos suelen ser buenos realizando tareas de una en una. Por eso separamos en dos pasos el filtrado de contenido relevante y el de extracción de ese contenido.
- Puedes usar cualquier proveedor y canal: vía API (como Groq/Gemini) o en local vía HuggingFce teniendo en cuenta las limitaciones y ventajas de cada uno, rate limits del modelo elegido, tamaño... 
- Tener en cuenta qué cantidad de tokens/información/tareas le estamos pasando al LLM en cada llamada comparándolo con los límites que tenemos para el LLM elegido
- Se valorará **justificar cada elección** como: cada time.stop() añadido, cada lote de filas de DataFrame que pasemos en cada llamada... etc ajustándonos como digo a esos límites que tenemos.

In [29]:
import pandas as pd

In [30]:
data = pd.read_csv('videogames_reviews.csv')
data

,Unnamed: 0,Contenido,Valoración,Recomendado_binario
0,0,2 marzo so bad,No recomendado,0
1,1,10 febrero actualmente recomiendo juego contab...,No recomendado,0
2,2,9 febrero increíblemente gracioso ver cómo cdp...,No recomendado,0
3,3,the world in this game is extremely static the...,No recomendado,0
4,4,zero replayability i finished this game in abo...,No recomendado,0
...,...,...,...,...
19995,19995,si,Recomendado,1
19996,19996,cojonudo,Recomendado,1
19997,19997,reostia historia guapisima graficos impresiona...,Recomendado,1
19998,19998,basicamente sublime obra maestra,Recomendado,1


In [31]:
# Análisis exploratorio básico
print("Información básica del dataframe:")
print(f"Forma del dataframe: {data.shape}")
print(f"Columnas: {list(data.columns)}")
print("\nTipos de datos:")
print(data.dtypes)
print("\nPrimeras filas:")
data.head()

Información básica del dataframe:
Forma del dataframe: (20000, 4)
Columnas: ['Unnamed: 0', 'Contenido', 'Valoración', 'Recomendado_binario']

Tipos de datos:
Unnamed: 0              int64
Contenido              object
Valoración             object
Recomendado_binario     int64
dtype: object

Primeras filas:


,Unnamed: 0,Contenido,Valoración,Recomendado_binario
0,0,2 marzo so bad,No recomendado,0
1,1,10 febrero actualmente recomiendo juego contab...,No recomendado,0
2,2,9 febrero increíblemente gracioso ver cómo cdp...,No recomendado,0
3,3,the world in this game is extremely static the...,No recomendado,0
4,4,zero replayability i finished this game in abo...,No recomendado,0


In [32]:
# Análisis de valores nulos y únicos
print("Valores nulos por columna:")
print(data.isnull().sum())
print("\nValores únicos en 'Valoración':")
print(data['Valoración'].value_counts())
print("\nEstadísticas de la columna 'Recomendado_binario':")
print(data['Recomendado_binario'].value_counts())

Valores nulos por columna:
Unnamed: 0               0
Contenido              288
Valoración               0
Recomendado_binario      0
dtype: int64

Valores únicos en 'Valoración':
Valoración
No recomendado    10000
Recomendado       10000
Name: count, dtype: int64

Estadísticas de la columna 'Recomendado_binario':
Recomendado_binario
0    10000
1    10000
Name: count, dtype: int64


## **PASO 1**
Obtener los 100 comentarios con más texto para análisis posterior.

In [ ]:
# Preprocesamiento simplificado: Top 100 comentarios con más texto
print("Preprocesando datos para obtener los 100 comentarios más largos...")

# Eliminar filas con contenido nulo
df_limpio = data.dropna(subset=['Contenido']).copy()
print(f"Comentarios válidos (sin nulos): {len(df_limpio)}")

# Calcular longitud de caracteres
df_limpio['longitud_caracteres'] = df_limpio['Contenido'].str.len()

# Seleccionar los 100 comentarios más largos
df_top_100 = df_limpio.nlargest(100, 'longitud_caracteres').reset_index(drop=True)

# Eliminar columnas innecesarias
if 'Unnamed: 0' in df_top_100.columns:
    df_top_100 = df_top_100.drop('Unnamed: 0', axis=1)

print(f"\nDataset final: {len(df_top_100)} comentarios")
print(f"Longitud promedio: {df_top_100['longitud_caracteres'].mean():.1f} caracteres")
print(f"Longitud mínima: {df_top_100['longitud_caracteres'].min()} caracteres")
print(f"Longitud máxima: {df_top_100['longitud_caracteres'].max()} caracteres")

print("\nDistribución de valoraciones en el top 100:")
print(df_top_100['Valoración'].value_counts())

# Mostrar el dataframe resultante
df_top_100

Preprocesando datos para obtener los 100 comentarios más largos...
Comentarios válidos (sin nulos): 19712

Dataset final: 100 comentarios
Longitud promedio: 6772.1 caracteres
Longitud mínima: 5421 caracteres
Longitud máxima: 7972 caracteres

Distribución de valoraciones en el top 100:
Valoración
No recomendado    65
Recomendado       35
Name: count, dtype: int64

Dataset final: 100 comentarios
Longitud promedio: 6772.1 caracteres
Longitud mínima: 5421 caracteres
Longitud máxima: 7972 caracteres

Distribución de valoraciones en el top 100:
Valoración
No recomendado    65
Recomendado       35
Name: count, dtype: int64


,Contenido,Valoración,Recomendado_binario,longitud_caracteres
0,suiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiii...,Recomendado,1,7972
1,this was probably my first preorder i felt tha...,Recomendado,1,7662
2,oh well admittedly its difficult for to write ...,No recomendado,0,7638
3,oh well admittedly its difficult for to write ...,No recomendado,0,7638
4,i know many will handwave away any criticisms ...,No recomendado,0,7609
...,...,...,...,...
95,4 febrero i was not seated in the glorious hyp...,Recomendado,1,5607
96,cyberpunk 2077cybermeme 2077what can there be ...,Recomendado,1,5573
97,brujo geralt riviatras recuperar memoria pasad...,Recomendado,1,5516
98,after hearing all the fuss about skyrim for ma...,No recomendado,0,5458


## **Paso 2: Continúa...**